# Fine-Tuning Laya on ambiguity-chaser's testability schema (single T4)

Fine-tunes **Laya** (`convaiinnovations/laya`, 421M params, English checkpoint) on our own 4-criterion testability-scoring schema, adapted from the vendor's [`laya_finetune_typed_decisions_2xT4_kaggle.ipynb`](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb) reference notebook.

**Differences from the vendor reference, deliberate:**
- **Single T4, no DDP.** Our dataset (168 train / 39 held-out items) is ~25x smaller than the vendor's typed-decisions benchmark (1,200/400 cases) — one GPU is plenty, so `torch.distributed`/`torchrun` is stripped out entirely.
- **Our own data, not `LocalLLaMA/typed-decisions`.** Loaded from 3 files uploaded this session: `finetune_dataset_claude_v1.jsonl` (train, Claude-authored text+labels), `finetune_holdout_texts.txt` + `finetune_holdout_deepseek_labels.jsonl` (held-out eval, labels from the real production DeepSeek judge — a different source than training labels, on purpose).
- **Our own question schema, not theirs.** 4 criteria: `has_measurable_condition`, `has_vague_qualitative_language`, `has_ambiguous_scope` as `noul`; `missing_precondition` as `choice` (switched from `noul` per the zero-shot finding that `noul` label-anchors on negated phrasing — see `HANDOVER.md`).
- **Crisp 0/1 targets, not soft teacher-agreement distributions.** The vendor's `gold` labels come from multi-teacher agreement counts; ours are single hand-authored booleans, so targets are one-hot rather than a probability vector.

Baselines already measured (don't re-run): **zero-shot Laya 57-61% overall**, **DeepSeek baseline 96-100%** (see `HANDOVER.md` section 1). This run's goal is to see how close fine-tuning closes that gap.

In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" safetensors huggingface_hub tabulate
# pandas deliberately NOT upgraded here -- Colab pins pandas==2.2.3 for its own
# internals (upload/download widgets included); a bare `-U pandas` pulled 3.0.6
# and broke that pin on the first attempt. Keep whatever Colab shipped.
import laya, transformers, torch, pandas
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)
print("Pandas version      :", pandas.__version__)
print("CUDA available      :", torch.cuda.is_available())

## 2. Upload our data files

Upload all 3 files from `ambiguity-chaser-laya/scratch/` when prompted:
- `finetune_dataset_claude_v1.jsonl` (train)
- `finetune_holdout_texts.txt` (held-out texts)
- `finetune_holdout_deepseek_labels.jsonl` (held-out labels, from the real DeepSeek judge)

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

REQUIRED = {
    "finetune_dataset_claude_v1.jsonl",
    "finetune_holdout_texts.txt",
    "finetune_holdout_deepseek_labels.jsonl",
}
missing = REQUIRED - set(uploaded.keys())
assert not missing, f"Missing required file(s): {missing}"
print("All 3 required files present.")

## 3. Fetch tokenizer/config and build training items

Same `build_sequence` our zero-shot scripts already exercise, but building items directly from our own JSONL labels (crisp booleans, one-hot targets) instead of a teacher-agreement distribution.

## 3b. Add targeted examples for the two gaps found in the first run

First run (15 epochs on the original 168 items) got 86% overall / 77% on `missing_precondition`, up from 57%/29% zero-shot. Error analysis on the 22 held-out misses found two confidently-wrong, systematic gaps rather than noise:
- `has_measurable_condition` under-detects **concrete state changes with no literal number** (e.g. "must be reviewed by a human agent") — training set skewed toward numeric thresholds.
- `has_ambiguous_scope` under-detects **vague operational verbs** (archive, flag, merge, route...) whose action is underspecified even when a trigger/number is present.

31 new Claude-authored examples targeting exactly those two gaps, `scratch/finetune_dataset_claude_v2_additions.jsonl` in the worktree — none overlap with the held-out eval texts (checked against the error-analysis printout above). Embedded inline below rather than a second upload round-trip.

In [ ]:
import json

v2_additions = """{"text": "An order must be marked as shipped once the carrier confirms pickup.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A patient's allergy list must be updated whenever a new prescription is entered.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A smart lock must relock automatically after the door is closed.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A support ticket must be reassigned when the original agent goes offline.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A deployment must be rolled back if the health check fails.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A flagged comment must be hidden from public view pending moderator review.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A player's inventory must be restored to its pre-match state after the match ends.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A shipment must be marked as delayed when it misses its scheduled departure.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A student's course progress must be saved when they exit a lesson.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A permit application must be forwarded to a reviewing officer once submitted.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A wire transfer must be held for compliance review when the recipient account is newly added.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "An employee's access badge must be deactivated on their last working day.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A meeting recording must stop automatically when the host leaves the call.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A document must be locked for editing while another user has it open.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A downloaded file must be removed from local storage after the app is uninstalled.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A user's session must be terminated if their account is flagged as compromised.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "An API key must be revoked once it is reported as leaked.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A gift card balance must be transferred to a new card.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": true}
{"text": "Expired promotional codes must be removed from the active codes list.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": false, "missing_precondition": false}
{"text": "A cancelled order must be processed by the fulfillment team.", "has_measurable_condition": false, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "The system must archive user accounts that have been inactive for 12 months.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "The moderation queue must flag messages containing prohibited content.", "has_measurable_condition": false, "has_vague_qualitative_language": true, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "When a build fails twice in a row, the pipeline must escalate the incident to the on-call engineer.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "Duplicate customer records must be merged automatically.", "has_measurable_condition": false, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": true}
{"text": "The system must clean up temporary files after a job completes.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "Stale cache entries must be cleared during off-peak hours.", "has_measurable_condition": false, "has_vague_qualitative_language": true, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "Inbound support emails must be routed to the appropriate department.", "has_measurable_condition": false, "has_vague_qualitative_language": true, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "All PII fields must be sanitized before being logged.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "Inventory counts must be normalized across all warehouse systems nightly.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": false}
{"text": "Failed login attempts must be audited for security review.", "has_measurable_condition": false, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": true}
{"text": "Duplicate invoice line items must be consolidated before the invoice is finalized.", "has_measurable_condition": true, "has_vague_qualitative_language": false, "has_ambiguous_scope": true, "missing_precondition": false}
"""

with open("finetune_dataset_claude_v2_additions.jsonl", "w", encoding="utf-8") as f:
    f.write(v2_additions)

n_v2 = sum(1 for line in v2_additions.strip().splitlines())
print(f"Wrote {n_v2} new examples.")

# Sanity check: none of the new texts overlap with the held-out eval file.
holdout_texts = set()
with open("finetune_holdout_deepseek_labels.jsonl", encoding="utf-8") as f:
    for line in f:
        holdout_texts.add(json.loads(line)["text"])
v2_texts = {json.loads(line)["text"] for line in v2_additions.strip().splitlines()}
overlap = v2_texts & holdout_texts
assert not overlap, f"New examples leak into held-out eval set: {overlap}"
print("No overlap with held-out eval set -- confirmed.")

with open("finetune_dataset_claude_v1.jsonl", encoding="utf-8") as f_v1, \
     open("finetune_dataset_claude_v2.jsonl", "w", encoding="utf-8") as f_v2:
    f_v2.write(f_v1.read())
    f_v2.write(v2_additions)

n_total = sum(1 for _ in open("finetune_dataset_claude_v2.jsonl", encoding="utf-8"))
print(f"Combined training set: {n_total} texts.")

In [ ]:
import os, json, random
import torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, QTYPES

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

# Wording verbatim from scratch/scratch_laya_scoring.py (noul, 3 criteria) and
# scratch/scratch_laya_choice_variant.py (choice, missing_precondition) -- the
# already-tested formulations, not reworded here.
CRITERIA = [
    "has_measurable_condition",
    "has_vague_qualitative_language",
    "has_ambiguous_scope",
    "missing_precondition",
]

QUESTIONS = {
    "has_measurable_condition": {
        "type": "noul",
        "instructions": (
            "Does the requirement contain at least one measurable or verifiable "
            "condition -- a number, a named field/value, or a concrete state change?"
        ),
    },
    "has_vague_qualitative_language": {
        "type": "noul",
        "instructions": (
            "Does the requirement rely on a vague qualitative word with no "
            "quantifier (e.g. 'fast', 'appropriate', 'reasonable', 'secure', "
            "'user-friendly')?"
        ),
    },
    "has_ambiguous_scope": {
        "type": "noul",
        "instructions": "Could the requirement's expected outcome reasonably be read more than one way?",
    },
    "missing_precondition": {
        "type": "choice",
        "instructions": "Does the requirement fail to state an explicit trigger or precondition for when it applies?",
        "criteria": {
            "A": "yes, the requirement fails to state an explicit trigger or precondition",
            "B": "no, the requirement does state an explicit trigger or precondition",
        },
    },
}


def build_training_item(tok, text, crit_name, label_bool):
    qdef = QUESTIONS[crit_name]
    q = {"t": qdef["type"], "ins": qdef["instructions"], "crit": qdef.get("criteria")}
    seq, markers = build_sequence(tok, text, q, cfg["max_len"], cfg["head_max_len"])
    if qdef["type"] == "noul":
        # render_options order for noul is always [false, true]
        target = [0.0, 1.0] if label_bool else [1.0, 0.0]
        label = 1 if label_bool else 0
    else:  # choice, keys in declared order: A, B
        target = [1.0, 0.0] if label_bool else [0.0, 1.0]
        label = 0 if label_bool else 1
    if len(markers) != len(target):
        return None
    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES[qdef["type"]],
        "target": target,
        "label": label,
        "criterion": crit_name,
        "text": text,
    }


TRAIN_FILE = "finetune_dataset_claude_v2.jsonl"  # v1 + the 31 gap-targeted additions

train_items = []
with open(TRAIN_FILE, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        text = row["text"]
        for crit in CRITERIA:
            it = build_training_item(tok, text, crit, bool(row[crit]))
            if it is not None:
                train_items.append(it)

print(f"Built {len(train_items)} training sequences from "
      f"{sum(1 for _ in open(TRAIN_FILE, encoding='utf-8'))} requirement texts "
      f"x {len(CRITERIA)} criteria.")
torch.save(train_items, "train_items.pt")

## 4. Single-GPU RLCD training

Same algorithm as the vendor's `train_ddp.py` (RLCD: GRPO-style group baseline, strictly-proper-scoring-rule reward, encoder/head differential LR, cosine schedule, sigma decay, post-training temperature calibration) — `torch.distributed`/`DDP` stripped since we have one GPU, run in-process rather than via `torchrun`.

**Round 3 (this run): EPOCHS dropped 15 → 8.** Round 2 (15 epochs, 199-item combined set) reached loss 0.0 by epoch 14 and both calibration temperatures clamped at the fitter's max (10.0) — genuine overfitting, not just a marginal accuracy number. Round 2's own per-epoch log had loss already at 0.12 by epoch 8. Testing whether stopping there keeps most of the accuracy gain with a non-clamped, more honest calibration — single variable changed, everything else identical to round 2.

In [ ]:
import os, time, json, random
import torch
from safetensors.torch import save_file
from laya.common import build_model, proper_reward


def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
        "target": target, "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
    }


def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())


device = torch.device("cuda")
OUTPUT_DIR = "/content/laya_finetuned_testability"

cfg["gradient_checkpointing"] = True
model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
from safetensors.torch import load_file
weights = load_file(os.path.join(model_dir, "model.safetensors"))
model.load_state_dict(weights, strict=True)
model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.head_checkpointing = True
model.to(device)
model.train()

all_items = torch.load("train_items.pt", weights_only=False)

# Calibration slice held out of TRAINING only -- not the real held-out eval set.
CALIB_FRACTION = 0.10
order = list(range(len(all_items)))
random.Random(20260926).shuffle(order)
n_calib = max(10, int(len(all_items) * CALIB_FRACTION))
calib_items = [all_items[i] for i in sorted(order[:n_calib])]
train_items_split = [all_items[i] for i in sorted(order[n_calib:])]

# EPOCHS dropped 15 -> 8: round 2 (15 epochs, 199-item combined set) hit loss
# 0.0 by epoch 14 and both calibration temperatures clamped at the fitter's
# max (10.0) -- a real overfitting signal, not just a slightly-worse number.
# Round 2's own per-epoch log showed loss already down to 0.12 by epoch 8,
# well before the later full memorization -- testing whether stopping there
# keeps most of the accuracy gain with a saner (non-clamped) calibration.
# Single variable changed, nothing else.
EPOCHS = 8
MICRO_BATCH = 8
GRAD_ACCUM = 2
GROUP_SIZE = 4
LR_ENCODER = 2.5e-5
LR_HEAD = 1.0e-4
SIGMA_START = 0.4
SIGMA_END = 0.1

enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
optimizer = torch.optim.AdamW([
    {"params": enc_params, "lr": LR_ENCODER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=0.01)

total_updates = (len(train_items_split) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=True)

print(f"Training: {len(train_items_split)} items ({len(calib_items)} held out for calibration) | {EPOCHS} epochs")
t0 = time.time()

for epoch in range(EPOCHS):
    random.seed(42 + epoch)
    random.shuffle(train_items_split)
    epoch_loss, n_batches = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    accum_step = 0
    progress = epoch / max(1, EPOCHS - 1)
    sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress

    for b_idx in range(0, len(train_items_split), MICRO_BATCH):
        chunk = train_items_split[b_idx:b_idx + MICRO_BATCH]
        if not chunk:
            continue
        batch = collate_train_batch(chunk, tok.pad_token_id)

        with torch.autocast("cuda", dtype=torch.float16):
            logits, act = model(
                batch["input_ids"].to(device), batch["attention_mask"].to(device),
                batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                batch["qtype"].to(device),
            )
        logits = logits.float()
        mask = batch["marker_mask"].to(device)
        k = mask.sum(-1, keepdim=True).float()
        target = batch["target"].to(device)

        eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
        eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
        z = logits.detach().unsqueeze(0) + eps
        q = torch.softmax(z.masked_fill(~mask, -1e4), -1)

        with torch.no_grad():
            r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
            adv = r - r.mean(0, keepdim=True)
            adv = adv / (adv.std() + 1e-6)

        logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
        loss_rl = -(adv * logp).mean()
        loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
        loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

        scaler.scale(loss).backward()
        accum_step += 1
        if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(train_items_split):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        epoch_loss += loss.item() * GRAD_ACCUM
        n_batches += 1

    print(f"=== Epoch {epoch+1}/{EPOCHS} | {time.time()-t0:.1f}s elapsed | "
          f"avg loss {epoch_loss/max(1,n_batches):.4f} | last reward {r.mean().item():.3f} ===")

    ckpt_dir = os.path.join(OUTPUT_DIR, "checkpoint_latest")
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
    save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
    tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))

print(f"\nTraining done in {time.time()-t0:.1f}s. Fitting calibration temperatures...")
del optimizer, scaler, scheduler
torch.cuda.empty_cache()
model.eval()
calib_preds = []
with torch.no_grad():
    for c_idx in range(0, len(calib_items), 16):
        c_chunk = calib_items[c_idx:c_idx + 16]
        cb = collate_train_batch(c_chunk, tok.pad_token_id)
        with torch.autocast("cuda", dtype=torch.float16):
            l_sub, _ = model(
                cb["input_ids"].to(device), cb["attention_mask"].to(device),
                cb["marker_pos"].to(device), cb["marker_mask"].to(device),
                cb["qtype"].to(device),
            )
        l_np = l_sub.float().cpu().numpy()
        for row_i, it in enumerate(c_chunk):
            kk = len(it["markers"])
            calib_preds.append((it["qtype"], l_np[row_i, :kk], it["target"]))

fitted_temps = [1.2, 1.2, 1.2]  # index order matches QTYPES: choice, score, noul
for qt in range(3):
    sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
    if sel:
        fitted_temps[qt] = fit_one_temp(sel)
print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])

os.makedirs(OUTPUT_DIR, exist_ok=True)
sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
save_file(sd, os.path.join(OUTPUT_DIR, "model.safetensors"))
model.encoder.config.save_pretrained(os.path.join(OUTPUT_DIR, "encoder"))
tok.save_pretrained(os.path.join(OUTPUT_DIR, "tokenizer"))
cfg["fine_tuned"] = True
cfg["model_name"] = "laya-testability-ambiguity-chaser"
cfg["temperature"] = fitted_temps
cfg.pop("temperature_by_options", None)
with open(os.path.join(OUTPUT_DIR, "rl_agent_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)
print(f"Model saved to {OUTPUT_DIR}")

## 5. Evaluate against the held-out DeepSeek-labeled set

This is the real "did it work" check — `finetune_holdout_deepseek_labels.jsonl` labels come from the actual production judge (`call_deepseek_json` + `validate_response` against `testability_score_v1.txt`), a different source than the training labels. Compare against the already-measured baselines: zero-shot Laya (57-61% overall, `missing_precondition` worst at 29-43%) and DeepSeek (96-100%).

In [ ]:
import time, json
import numpy as np
import pandas as pd
from laya.common import ece_score, answer_confidence

model.eval()

holdout_rows = []
with open("finetune_holdout_deepseek_labels.jsonl", encoding="utf-8") as f:
    for line in f:
        holdout_rows.append(json.loads(line))
print(f"Loaded {len(holdout_rows)} held-out DeepSeek-labeled rows.")

per_crit_correct = {c: 0 for c in CRITERIA}
per_crit_total = {c: 0 for c in CRITERIA}
all_confs, all_corrects = [], []
latencies_ms = []

with torch.no_grad():
    for row in holdout_rows:
        text = row["text"]
        for crit in CRITERIA:
            if crit not in row:
                continue
            gold = bool(row[crit])
            qdef = QUESTIONS[crit]
            q = {"t": qdef["type"], "ins": qdef["instructions"], "crit": qdef.get("criteria")}
            seq, markers = build_sequence(tok, text, q, cfg["max_len"], cfg["head_max_len"])
            item = {
                "ids": seq, "markers": markers, "qtype": QTYPES[qdef["type"]],
                "label": 0,
            }
            batch = collate_train_batch([{**item, "target": [0.0] * len(markers)}], tok.pad_token_id)

            t0 = time.perf_counter()
            with torch.autocast("cuda", dtype=torch.float16):
                logits, _ = model(
                    batch["input_ids"].to(device), batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                    batch["qtype"].to(device),
                )
            dt_ms = (time.perf_counter() - t0) * 1000
            latencies_ms.append(dt_ms)

            k = len(markers)
            z = logits[0, :k].float().cpu().numpy()
            temp = fitted_temps[QTYPES[qdef["type"]]]
            p = np.exp(z / temp) / np.exp(z / temp).sum()

            if qdef["type"] == "noul":
                pred_bool = bool(p[1] >= p[0])  # index 1 = true
            else:  # choice: index 0 = "A" = missing_precondition True
                pred_bool = bool(p[0] >= p[1])

            conf = answer_confidence(p, k)
            is_correct = float(pred_bool == gold)
            per_crit_total[crit] += 1
            per_crit_correct[crit] += is_correct
            all_confs.append(conf)
            all_corrects.append(is_correct)

overall_correct = sum(per_crit_correct.values())
overall_total = sum(per_crit_total.values())
ft_ece = ece_score(np.array(all_confs), np.array(all_corrects))
ft_latency_p50 = float(np.percentile(latencies_ms, 50))

print("=== Fine-tuned Laya accuracy vs. real DeepSeek-labeled held-out set ===")
for crit in CRITERIA:
    t = per_crit_total[crit]
    c = per_crit_correct[crit]
    print(f"  {crit:32s} {c:.0f}/{t} ({100*c/t:.0f}%)")
print(f"  {'OVERALL':32s} {overall_correct:.0f}/{overall_total} ({100*overall_correct/overall_total:.0f}%)")
print(f"  ECE: {ft_ece:.3f}")
print(f"  latency p50: {ft_latency_p50:.2f} ms")

comparison = pd.DataFrame([
    {"Model": "Laya zero-shot (GPU)", "Overall Acc": "57%", "missing_precondition Acc": "29%", "Source": "HANDOVER.md sec.1"},
    {"Model": "DeepSeek baseline", "Overall Acc": "96-100%", "missing_precondition Acc": "n/a (not broken out)", "Source": "HANDOVER.md sec.1"},
    {"Model": "Laya fine-tuned (this run)",
     "Overall Acc": f"{100*overall_correct/overall_total:.0f}%",
     "missing_precondition Acc": f"{100*per_crit_correct['missing_precondition']/per_crit_total['missing_precondition']:.0f}%",
     "Source": "this run"},
])
print("\n" + comparison.to_markdown(index=False))

In [ ]:
wrong = []
with torch.no_grad():
    for row in holdout_rows:
        text = row["text"]
        for crit in CRITERIA:
            if crit not in row:
                continue
            gold = bool(row[crit])
            qdef = QUESTIONS[crit]
            q = {"t": qdef["type"], "ins": qdef["instructions"], "crit": qdef.get("criteria")}
            seq, markers = build_sequence(tok, text, q, cfg["max_len"], cfg["head_max_len"])
            item = {"ids": seq, "markers": markers, "qtype": QTYPES[qdef["type"]], "label": 0}
            batch = collate_train_batch([{**item, "target": [0.0] * len(markers)}], tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                logits, _ = model(
                    batch["input_ids"].to(device), batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                    batch["qtype"].to(device),
                )
            k = len(markers)
            z = logits[0, :k].float().cpu().numpy()
            temp = fitted_temps[QTYPES[qdef["type"]]]
            p = np.exp(z / temp) / np.exp(z / temp).sum()
            if qdef["type"] == "noul":
                pred_bool = bool(p[1] >= p[0])
                conf = float(max(p))
            else:
                pred_bool = bool(p[0] >= p[1])
                conf = float(max(p))
            if pred_bool != gold:
                wrong.append((crit, text, gold, pred_bool, conf, row.get("reasoning", "")))

print(f"{len(wrong)} wrong out of 156 checks\n")
for crit, text, gold, pred, conf, reasoning in wrong:
    print(f"[{crit}] conf={conf:.2f} pred={pred} gold={gold}")
    print(f"  text: {text}")
    print(f"  deepseek reasoning: {reasoning[:200]}")
    print()

## 6. Download the checkpoint

Zips `OUTPUT_DIR` and triggers a browser download back to your machine.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/laya_finetuned_testability", "zip", OUTPUT_DIR)
print(f"Zipped to {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
files.download(zip_path)

In [ ]:
# Fallback only -- Pratham's preference is the direct browser download above;
# it's flaky at ~1.5GB (silently no-ops sometimes) but works on a retry more
# often than not. Use this Drive-mount path only if repeated retries fail.
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = "/content/drive/MyDrive/laya_finetuned_testability.zip"
shutil.copy("/content/laya_finetuned_testability.zip", dest)
print(f"Copied to {dest} -- download it from drive.google.com from there.")

## 7. (Optional) Push to Hugging Face Hub

**Not run by default** — whether to publish this checkpoint is Pratham's call (per `HANDOVER.md` step 8), not decided here. Set `PUSH_TO_HF = True` below only after that decision is made, and add `HF_TOKEN` to Colab's Secrets (key icon in the left sidebar) first.

In [ ]:
PUSH_TO_HF = False  # flip only after Pratham decides to publish

if PUSH_TO_HF:
    from google.colab import userdata
    from huggingface_hub import HfApi

    token = userdata.get("HF_TOKEN")
    NEW_REPO = "convaiinnovations/laya-ambiguity-chaser-testability"  # placeholder -- confirm target repo before running
    api = HfApi(token=token)
    api.create_repo(NEW_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=NEW_REPO, repo_type="model",
                       commit_message="Fine-tuned on ambiguity-chaser testability schema")
    print(f"Pushed to https://huggingface.co/{NEW_REPO}")
else:
    print("Skipped -- PUSH_TO_HF is False.")